# Projeto SBD — Etapa 1 | Exploração inicial da base Olist

**UFPB · Sistemas de Banco de Dados · Entrega: 25/11/2026**  
**Equipe:** Ana Beatriz · Maria Clara · Ayra  
**Fonte:** [Brazilian E-Commerce Public Dataset by Olist (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

## Objetivo desta etapa
Caracterizar os nove arquivos do dataset, examinar variáveis, qualidade e possíveis relacionamentos para justificar a escolha da base. **A modelagem relacional final, a importação definitiva para PostgreSQL e o CRUD ficam para as próximas etapas.**

### Estrutura usada no repositório
```text
gestao-e-analise-ecommerce/
├── data/
│   └── raw/
│       ├── olist_customers_dataset.csv
│       ├── olist_orders_dataset.csv
│       └── ... (demais CSVs da Olist)
├── docs/
├── notebooks/
│   └── Exploracao_Inicial_Olist_Etapa1.ipynb  ← este arquivo
├── sql/
├── src/
├── .gitignore
└── requirements.txt
```
Os resultados são calculados **a partir dos arquivos locais**. Nenhuma estatística é inventada. Os CSVs da base devem ser mantidos fora do versionamento.

## 1. Preparação e leitura dos arquivos
A base deve estar em `data/raw/`, como na organização do repositório. O notebook identifica a raiz do projeto tanto quando o VS Code executa a partir de `notebooks/` quanto quando executa a partir da pasta principal.

**Dependências:** `pandas`, `ipykernel` e `IPython` (instaláveis com `pip install pandas ipykernel`). A leitura utiliza `dtype=str` para preservar os zeros iniciais dos prefixos de CEP.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 100)

ARQUIVOS = {
    "clientes": "olist_customers_dataset.csv",
    "pedidos": "olist_orders_dataset.csv",
    "itens": "olist_order_items_dataset.csv",
    "produtos": "olist_products_dataset.csv",
    "vendedores": "olist_sellers_dataset.csv",
    "pagamentos": "olist_order_payments_dataset.csv",
    "avaliacoes": "olist_order_reviews_dataset.csv",
    "geolocalizacao": "olist_geolocation_dataset.csv",
    "traducao_categorias": "product_category_name_translation.csv",
}

def encontrar_raiz_projeto():
    atual = Path.cwd().resolve()
    for candidata in [atual, *atual.parents]:
        pasta_csv = candidata / "data" / "raw"
        if pasta_csv.is_dir() and (candidata / "notebooks").is_dir():
            return candidata
    raise FileNotFoundError(
        "Não localizei a estrutura do repositório. Abra a pasta gestao-e-analise-ecommerce "
        "no VS Code e mantenha os CSVs dentro de data/raw/."
    )

RAIZ_PROJETO = encontrar_raiz_projeto()
pasta = RAIZ_PROJETO / "data" / "raw"
presentes = {nome: (pasta / arquivo).is_file() for nome, arquivo in ARQUIVOS.items()}

print("Raiz do repositório:", RAIZ_PROJETO)
print("Pasta dos dados:", pasta)
display(pd.DataFrame({"arquivo": list(ARQUIVOS.values()), "encontrado": list(presentes.values())}))

if not all(presentes.values()):
    faltando = [ARQUIVOS[nome] for nome, existe in presentes.items() if not existe]
    raise FileNotFoundError("Arquivos não encontrados em data/raw/: " + ", ".join(faltando))

Raiz do repositório: C:\Users\reisc\Desktop\gestao-e-analise-ecommerce
Pasta dos dados: C:\Users\reisc\Desktop\gestao-e-analise-ecommerce\data\raw


,arquivo,encontrado
0,olist_customers_dataset.csv,True
1,olist_orders_dataset.csv,True
2,olist_order_items_dataset.csv,True
3,olist_products_dataset.csv,True
4,olist_sellers_dataset.csv,True
5,olist_order_payments_dataset.csv,True
6,olist_order_reviews_dataset.csv,True
7,olist_geolocation_dataset.csv,True
8,product_category_name_translation.csv,True


## 2. Inventário dos arquivos
Contamos linhas e colunas sem carregar todos os CSVs simultaneamente em memória. `geolocation` tem mais de um milhão de linhas; por isso o notebook evita juntar essa tabela integralmente às demais.

In [2]:
def contar_registros(caminho):
    # Arquivos CSV da Olist têm cabeçalho e registros em linha única.
    # Para contagem robusta usa-se leitura em blocos.
    total = 0
    for bloco in pd.read_csv(caminho, chunksize=100_000, dtype=str, encoding="utf-8"):
        total += len(bloco)
    return total

inventario = []
for apelido, nome in ARQUIVOS.items():
    caminho = pasta / nome
    amostra = pd.read_csv(caminho, nrows=5, dtype=str, encoding="utf-8")
    inventario.append({"arquivo": nome, "registros": contar_registros(caminho),
                       "colunas": len(amostra.columns), "tamanho_MB": round(caminho.stat().st_size / 1024**2, 2)})
resumo_arquivos = pd.DataFrame(inventario)
display(resumo_arquivos)
print(f"Arquivos identificados: {len(resumo_arquivos)}")

,arquivo,registros,colunas,tamanho_MB
0,olist_customers_dataset.csv,99441,5,8.62
1,olist_orders_dataset.csv,99441,8,16.84
2,olist_order_items_dataset.csv,112650,7,14.72
3,olist_products_dataset.csv,32951,9,2.27
4,olist_sellers_dataset.csv,3095,4,0.17
5,olist_order_payments_dataset.csv,103886,5,5.51
6,olist_order_reviews_dataset.csv,99224,7,13.78
7,olist_geolocation_dataset.csv,1000163,5,58.44
8,product_category_name_translation.csv,71,2,0.00


Arquivos identificados: 9


## 3. Dicionário inicial das variáveis
Listamos todas as colunas de cada arquivo. O tipo abaixo é o **tipo observado na leitura exploratória** (`string`), não uma definição do tipo SQL final. A modelagem e a escolha de tipos SQL ficam para a segunda entrega.

In [3]:
descricao = {
    "clientes": "Identificação e localização dos clientes ligados aos pedidos",
    "pedidos": "Status e datas de compra, aprovação, envio e entrega",
    "itens": "Produtos e vendedores de cada pedido, preços e frete",
    "produtos": "Categorias, descrições e características físicas dos produtos",
    "vendedores": "Identificação e localização dos vendedores",
    "pagamentos": "Forma, parcelas e valores de pagamento dos pedidos",
    "avaliacoes": "Notas, comentários e datas de avaliações dos pedidos",
    "geolocalizacao": "Coordenadas e localidades associadas a prefixos de CEP",
    "traducao_categorias": "Tradução de nomes de categorias de produtos",
}
linhas = []
for apelido, nome in ARQUIVOS.items():
    colunas = pd.read_csv(pasta / nome, nrows=0).columns.tolist()
    for coluna in colunas:
        linhas.append({"arquivo": nome, "finalidade": descricao[apelido], "variavel": coluna})
dicionario = pd.DataFrame(linhas)
display(dicionario)

,arquivo,finalidade,variavel
0,olist_customers_dataset.csv,Identificação e localização dos clientes ligados aos pedidos,customer_id
1,olist_customers_dataset.csv,Identificação e localização dos clientes ligados aos pedidos,customer_unique_id
2,olist_customers_dataset.csv,Identificação e localização dos clientes ligados aos pedidos,customer_zip_code_prefix
3,olist_customers_dataset.csv,Identificação e localização dos clientes ligados aos pedidos,customer_city
4,olist_customers_dataset.csv,Identificação e localização dos clientes ligados aos pedidos,customer_state
5,olist_orders_dataset.csv,"Status e datas de compra, aprovação, envio e entrega",order_id
6,olist_orders_dataset.csv,"Status e datas de compra, aprovação, envio e entrega",customer_id
7,olist_orders_dataset.csv,"Status e datas de compra, aprovação, envio e entrega",order_status
8,olist_orders_dataset.csv,"Status e datas de compra, aprovação, envio e entrega",order_purchase_timestamp
9,olist_orders_dataset.csv,"Status e datas de compra, aprovação, envio e entrega",order_approved_at


## 4. Qualidade inicial — valores ausentes, vazios e duplicatas exatas
Os CSVs são lidos **como texto** para não eliminar zeros à esquerda. Cada tabela é analisada separadamente; `geolocalizacao` é processada em blocos para reduzir o uso de memória. **Duplicatas exatas** significam linhas inteiramente idênticas; **chaves repetidas** são avaliadas separadamente mais adiante.

Uma célula vazia ou com espaços também é tratada como ausência na análise de qualidade (além dos valores nulos).

In [4]:
def perfil_csv(caminho, chunksize=100_000):
    faltantes = None
    total = 0
    # Contagem exata de duplicatas por conteúdo é feita separadamente para os
    # menores arquivos. Na geolocalização informamos 'não calculado' por memória.
    for bloco in pd.read_csv(caminho, dtype=str, chunksize=chunksize, encoding="utf-8"):
        vazios = bloco.isna() | bloco.fillna("").apply(lambda c: c.str.strip().eq(""))
        soma = vazios.sum()
        faltantes = soma if faltantes is None else faltantes.add(soma, fill_value=0)
        total += len(bloco)
    return total, faltantes.astype(int)

faltas = []
qualidade = []
for apelido, nome in ARQUIVOS.items():
    caminho = pasta / nome
    total, ausentes = perfil_csv(caminho)
    duplicatas = None
    if apelido != "geolocalizacao":
        df_temp = pd.read_csv(caminho, dtype=str, encoding="utf-8")
        duplicatas = int(df_temp.duplicated().sum())
        del df_temp
    qualidade.append({"arquivo": nome, "registros": total,
                      "celulas_ausentes": int(ausentes.sum()),
                      "duplicatas_exatas": duplicatas})
    for coluna, qtd in ausentes.items():
        faltas.append({"arquivo": nome, "coluna": coluna, "ausentes_ou_vazios": int(qtd),
                       "percentual": round(100 * qtd / total, 2) if total else 0})

resumo_qualidade = pd.DataFrame(qualidade)
qualidade_colunas = pd.DataFrame(faltas)
display(resumo_qualidade)
print("Colunas com pelo menos um valor ausente ou vazio:")
display(qualidade_colunas.query("ausentes_ou_vazios > 0").sort_values(
    ["arquivo", "ausentes_ou_vazios"], ascending=[True, False]))
print("Observação: duplicatas exatas da geolocalização não foram contadas nesta etapa.")

,arquivo,registros,celulas_ausentes,duplicatas_exatas
0,olist_customers_dataset.csv,99441,0,0.0
1,olist_orders_dataset.csv,99441,4908,0.0
2,olist_order_items_dataset.csv,112650,0,0.0
3,olist_products_dataset.csv,32951,2448,0.0
4,olist_sellers_dataset.csv,3095,0,0.0
5,olist_order_payments_dataset.csv,103886,0,0.0
6,olist_order_reviews_dataset.csv,99224,145932,0.0
7,olist_geolocation_dataset.csv,1000163,0,NaN
8,product_category_name_translation.csv,71,0,0.0


Colunas com pelo menos um valor ausente ou vazio:


,arquivo,coluna,ausentes_ou_vazios,percentual
41,olist_order_reviews_dataset.csv,review_comment_title,87658,88.34
42,olist_order_reviews_dataset.csv,review_comment_message,58274,58.73
11,olist_orders_dataset.csv,order_delivered_customer_date,2965,2.98
10,olist_orders_dataset.csv,order_delivered_carrier_date,1783,1.79
9,olist_orders_dataset.csv,order_approved_at,160,0.16
21,olist_products_dataset.csv,product_category_name,610,1.85
22,olist_products_dataset.csv,product_name_lenght,610,1.85
23,olist_products_dataset.csv,product_description_lenght,610,1.85
24,olist_products_dataset.csv,product_photos_qty,610,1.85
25,olist_products_dataset.csv,product_weight_g,2,0.01


Observação: duplicatas exatas da geolocalização não foram contadas nesta etapa.


## 5. Possíveis chaves e identificadores repetidos
Esta checagem ajuda a **propor** chaves, mas não cria restrições SQL. Repetições em `order_id` na tabela de itens, pagamentos ou avaliações podem refletir relações 1:N e **não são automaticamente erros**. Em `itens`, o par (`order_id`, `order_item_id`) é uma chave candidata a verificar.

In [5]:
chaves_candidatas = {
    "clientes": ["customer_id"],
    "pedidos": ["order_id"],
    "itens": ["order_id", "order_item_id"],
    "produtos": ["product_id"],
    "vendedores": ["seller_id"],
    "pagamentos": ["order_id", "payment_sequential"],
    "avaliacoes": ["review_id"],
    "traducao_categorias": ["product_category_name"],
}
resultados_chaves = []
for apelido, colunas in chaves_candidatas.items():
    df = pd.read_csv(pasta / ARQUIVOS[apelido], dtype=str)
    if not set(colunas).issubset(df.columns):
        continue
    resultados_chaves.append({
        "arquivo": ARQUIVOS[apelido], "chave_proposta": " + ".join(colunas),
        "linhas": len(df), "chaves_nulas_ou_vazias": int((df[colunas].isna() | df[colunas].fillna("").apply(lambda c: c.str.strip().eq(""))).any(axis=1).sum()),
        "combinacoes_repetidas": int(df.duplicated(subset=colunas).sum())
    })
    del df
resultado_chaves = pd.DataFrame(resultados_chaves)
display(resultado_chaves)
print("A geolocalização não recebe chave candidata por CEP: prefixos de CEP se repetem.")

,arquivo,chave_proposta,linhas,chaves_nulas_ou_vazias,combinacoes_repetidas
0,olist_customers_dataset.csv,customer_id,99441,0,0
1,olist_orders_dataset.csv,order_id,99441,0,0
2,olist_order_items_dataset.csv,order_id + order_item_id,112650,0,0
3,olist_products_dataset.csv,product_id,32951,0,0
4,olist_sellers_dataset.csv,seller_id,3095,0,0
5,olist_order_payments_dataset.csv,order_id + payment_sequential,103886,0,0
6,olist_order_reviews_dataset.csv,review_id,99224,0,814
7,product_category_name_translation.csv,product_category_name,71,0,0


A geolocalização não recebe chave candidata por CEP: prefixos de CEP se repetem.


## 6. Relacionamentos possíveis entre os arquivos
Testamos se valores nas tabelas filhas possuem correspondência na tabela de referência. Não fazemos `merge` com a geolocalização, pois vários registros podem compartilhar o mesmo prefixo de CEP. Esta é uma verificação exploratória, não uma declaração de integridade referencial no PostgreSQL.

In [6]:
RELACOES = [
    ("pedidos", "customer_id", "clientes", "customer_id"),
    ("itens", "order_id", "pedidos", "order_id"),
    ("itens", "product_id", "produtos", "product_id"),
    ("itens", "seller_id", "vendedores", "seller_id"),
    ("pagamentos", "order_id", "pedidos", "order_id"),
    ("avaliacoes", "order_id", "pedidos", "order_id"),
    ("produtos", "product_category_name", "traducao_categorias", "product_category_name"),
]

def valores_coluna(apelido, coluna):
    return pd.read_csv(pasta / ARQUIVOS[apelido], usecols=[coluna], dtype=str)[coluna]

verificacoes = []
for filha, campo_filha, pai, campo_pai in RELACOES:
    origem = valores_coluna(filha, campo_filha)
    referencia = set(valores_coluna(pai, campo_pai).dropna())
    sem_referencia = origem.notna() & ~origem.isin(referencia)
    verificacoes.append({"tabela_origem": ARQUIVOS[filha], "campo": campo_filha,
                          "tabela_referencia": ARQUIVOS[pai],
                          "linhas_sem_correspondencia": int(sem_referencia.sum()),
                          "valores_nulos_na_origem": int(origem.isna().sum())})
relacionamentos = pd.DataFrame(verificacoes)
display(relacionamentos)

,tabela_origem,campo,tabela_referencia,linhas_sem_correspondencia,valores_nulos_na_origem
0,olist_orders_dataset.csv,customer_id,olist_customers_dataset.csv,0,0
1,olist_order_items_dataset.csv,order_id,olist_orders_dataset.csv,0,0
2,olist_order_items_dataset.csv,product_id,olist_products_dataset.csv,0,0
3,olist_order_items_dataset.csv,seller_id,olist_sellers_dataset.csv,0,0
4,olist_order_payments_dataset.csv,order_id,olist_orders_dataset.csv,0,0
5,olist_order_reviews_dataset.csv,order_id,olist_orders_dataset.csv,0,0
6,olist_products_dataset.csv,product_category_name,product_category_name_translation.csv,13,610


## 7. Geolocalização e CEPs — cuidados de modelagem
Na Olist, o arquivo de geolocalização pode conter **várias coordenadas por prefixo de CEP**. Isso impede supor uma relação 1:1 ou uma chave única baseada apenas no prefixo. Aqui verificamos a repetição dos prefixos sem realizar `JOIN` multiplicador de linhas. Também checamos a quantidade de dígitos dos prefixos de clientes e vendedores.

In [7]:
geo = pd.read_csv(pasta / ARQUIVOS["geolocalizacao"], usecols=["geolocation_zip_code_prefix"], dtype=str)
frequencia_cep = geo["geolocation_zip_code_prefix"].value_counts(dropna=False)
print("Registros de geolocalização:", len(geo))
print("Prefixos de CEP distintos:", geo["geolocation_zip_code_prefix"].nunique())
print("Prefixos com mais de um registro:", int((frequencia_cep > 1).sum()))

resultado_cep = []
for apelido, coluna in [("clientes", "customer_zip_code_prefix"),
                        ("vendedores", "seller_zip_code_prefix"),
                        ("geolocalizacao", "geolocation_zip_code_prefix")]:
    serie = valores_coluna(apelido, coluna)
    comprimento_errado = serie.notna() & ~serie.str.fullmatch(r"\d{5}", na=False)
    resultado_cep.append({"arquivo": ARQUIVOS[apelido],
                           "ceps_fora_do_padrao_5_digitos": int(comprimento_errado.sum()),
                           "ceps_com_zero_inicial": int(serie.fillna("").str.startswith("0").sum())})
display(pd.DataFrame(resultado_cep))
del geo

Registros de geolocalização: 1000163
Prefixos de CEP distintos: 19015
Prefixos com mais de um registro: 17972


,arquivo,ceps_fora_do_padrao_5_digitos,ceps_com_zero_inicial
0,olist_customers_dataset.csv,0,23995
1,olist_sellers_dataset.csv,0,1027
2,olist_geolocation_dataset.csv,0,245733


## 8. Exploração descritiva para orientar as perguntas do projeto
São **contagens e distribuições iniciais**, não uma análise comercial conclusiva. Receita, ticket médio, atrasos e relação com avaliações dependem de regras de negócio e junções que serão definidas nas etapas seguintes.

In [8]:
clientes = pd.read_csv(pasta / ARQUIVOS["clientes"], dtype=str)
pedidos = pd.read_csv(pasta / ARQUIVOS["pedidos"], dtype=str)
pagamentos = pd.read_csv(pasta / ARQUIVOS["pagamentos"], dtype=str)
avaliacoes = pd.read_csv(pasta / ARQUIVOS["avaliacoes"], dtype=str)

print("Clientes por UF (10 maiores):")
display(clientes["customer_state"].value_counts().head(10).rename_axis("UF").reset_index(name="registros"))
print("Situação dos pedidos:")
display(pedidos["order_status"].value_counts(dropna=False).rename_axis("status").reset_index(name="pedidos"))
print("Formas de pagamento (registros, não pedidos únicos):")
display(pagamentos["payment_type"].value_counts(dropna=False).rename_axis("tipo").reset_index(name="registros"))
print("Notas das avaliações (registros, não pedidos únicos):")
display(avaliacoes["review_score"].value_counts(dropna=False).sort_index().rename_axis("nota").reset_index(name="avaliacoes"))
print("Consumidores únicos identificados:", clientes["customer_unique_id"].nunique())
del clientes, pedidos, pagamentos, avaliacoes

Clientes por UF (10 maiores):


,UF,registros
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


Situação dos pedidos:


,status,pedidos
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


Formas de pagamento (registros, não pedidos únicos):


,tipo,registros
0,credit_card,76795
1,boleto,19784
2,voucher,5775
3,debit_card,1529
4,not_defined,3


Notas das avaliações (registros, não pedidos únicos):


,nota,avaliacoes
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


Consumidores únicos identificados: 96096


## 9. Síntese para a primeira entrega

**Proposta de objetivo:** estruturar um banco de dados relacional para organizar informações de uma operação de comércio eletrônico e apoiar consultas sobre vendas, entregas e experiência dos clientes.

**Possíveis perguntas para etapas posteriores:**
1. Quais categorias de produtos apresentam maior receita de itens vendidos?
2. Quais vendedores concentram mais itens e pedidos?
3. Como os pedidos se distribuem entre as unidades federativas?
4. Quais formas de pagamento são mais frequentes?
5. Qual a frequência de atrasos nas entregas, considerando a data estimada e a entrega real?
6. Como se distribuem as avaliações de pedidos entregues no prazo e fora dele?

**Justificativa:** a Olist é uma base pública, com múltiplos CSVs conectáveis por identificadores, apropriada para desenvolver um modelo com pelo menos cinco tabelas, importar dados, executar consultas SQL e construir posteriormente uma aplicação CRUD.

**Limitações e cuidados:**
- `customer_id` e `customer_unique_id` possuem significados diferentes; um consumidor pode aparecer em mais de um registro.
- Os arquivos de itens, pagamentos e avaliações podem ter múltiplas linhas associadas a um pedido; junções ingênuas podem duplicar valores monetários.
- Os prefixos de CEP devem permanecer texto, preservando zeros à esquerda.
- O conjunto de geolocalização não é um cadastro único de CEPs; será preciso definir tratamento específico se ele entrar no modelo.
- Ausência de valores nulos não garante ausência de erros semânticos; a interpretação de duplicatas depende da entidade.
- Os dados cobrem uma operação histórica, portanto não representam o comércio eletrônico brasileiro atual.

**Decisão final sobre escopo, regras de cálculo e modelagem:** deve ser validada em conjunto pelas três integrantes. O modelo relacional normalizado e as chaves definitivas são da segunda entrega (02/12).

**Cuidados de interpretação:** contagens da tabela de clientes correspondem a registros vinculados a pedidos, não necessariamente a indivíduos diferentes; pedidos podem possuir vários itens ou vários registros de pagamento/avaliação. Campos ausentes em datas de entrega podem decorrer do status do pedido. As decisões finais de chaves, integridade referencial e normalização dependem da etapa 2.

## 10. Exportação dos resumos (opcional)

Os resumos gerados são arquivos pequenos salvos em `docs/resultados_etapa1/`, para facilitar o preenchimento do documento da equipe. **Não são exportados os dados brutos.**

In [9]:
saida = RAIZ_PROJETO / "docs" / "resultados_etapa1"
saida.mkdir(exist_ok=True)
for nome, tabela in {
    "inventario.csv": resumo_arquivos,
    "dicionario_variaveis.csv": dicionario,
    "qualidade_arquivos.csv": resumo_qualidade,
    "qualidade_colunas.csv": qualidade_colunas,
    "chaves_candidatas.csv": resultado_chaves,
    "relacionamentos.csv": relacionamentos,
}.items():
    tabela.to_csv(saida / nome, index=False, encoding="utf-8-sig")
print("Resultados exportados para:", saida.resolve())

Resultados exportados para: C:\Users\reisc\Desktop\gestao-e-analise-ecommerce\docs\resultados_etapa1
